# Problem D — Emerging Risk Discovery: Visualizations

Replicates and extends all charts from the Streamlit **Problem D** page.

| Section | Charts |
|---------|--------|
| Risk Ranking | Horizontal bar chart (top N by risk score), growth-severity scatter |
| Trend Explorer | Report volume over time with PELT changepoint markers |
| Topic Browser | Report volume bar + severity trend area chart per topic |
| Extra EDA | Severity heatmap over time, changepoint summary |

**Pre-req:** Run `scripts/run_phase4.py` to generate `data/processed/emerging_risks.csv`.

In [1]:
import os
from pathlib import Path

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
print(f"Working directory: {root}")

Working directory: e:\OSFDA


In [2]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

PROCESSED = Path("data/processed")
print("Libraries loaded.")

Libraries loaded.


## 1. Load Emerging Risk Data

In [3]:
risks  = pd.read_csv(PROCESSED / "emerging_risks.csv")
trends = pd.read_parquet(PROCESSED / "topic_trends.parquet")

with open(PROCESSED / "topic_representations.json") as f:
    reps = json.load(f)
with open(PROCESSED / "topic_changepoints.json") as f:
    changepoints = json.load(f)

def _topic_label(row):
    name  = str(row.get("Name", row.get("Topic", "")))
    parts = name.split("_", 1)
    return parts[1].replace("_", " ").title() if len(parts) > 1 else name

def _get_keywords(topic_id):
    key = str(int(topic_id))
    val = reps.get(key, reps.get(int(topic_id), {}))
    if isinstance(val, dict):  return val.get("keywords", [])
    if isinstance(val, list):  return val
    return []

risks["label"]    = risks.apply(_topic_label, axis=1)
risks_sorted      = risks.sort_values("Risk_Score", ascending=False).reset_index(drop=True)

print(f"Topics:       {len(risks):,}")
print(f"Trend rows:   {len(trends):,}")
print(f"Periods:      {trends['period'].nunique()}")
print(f"With changepoints: {sum(1 for v in changepoints.values() if v)}")

Topics:       30
Trend rows:   180
Periods:      6
With changepoints: 2


## 2. Top Topics by Risk Score

In [4]:
top_n   = 15
df_top  = risks_sorted.head(top_n).copy()
df_top["Changepoint"] = df_top["Recent_Changepoint"].map({True: "Yes", False: "No"})

fig = px.bar(
    df_top,
    x="Risk_Score",
    y="label",
    orientation="h",
    color="Changepoint",
    color_discrete_map={"Yes": "#e74c3c", "No": "#3498db"},
    hover_data={"Growth_Ratio": ":.2f", "Avg_Severity": ":.2f", "Count": True},
    title=f"Top {top_n} Emerging Risk Topics by Risk Score\n(Risk = Growth × Severity × log(Count))",
    labels={"Risk_Score": "Risk Score", "label": "Topic"},
)
fig.update_yaxes(autorange="reversed")
fig.update_layout(height=max(400, top_n * 32), margin=dict(l=220, t=70, b=30))
fig.show()

## 3. Growth Ratio vs Average Severity Landscape

In [5]:
fig = px.scatter(
    risks_sorted,
    x="Growth_Ratio",
    y="Avg_Severity",
    size="Count",
    color="Risk_Score",
    color_continuous_scale="RdYlGn_r",
    hover_name="label",
    hover_data={"Risk_Score": ":.2f", "Count": True, "Recent_Changepoint": True},
    title="Topic Landscape: Growth Ratio × Avg Severity (bubble size = report count)",
    labels={
        "Growth_Ratio": "Growth Ratio (recent / baseline)",
        "Avg_Severity": "Avg Incident Severity (0–3)",
    },
)
fig.add_vline(x=1.0, line_dash="dash", line_color="gray", annotation_text="No growth")
fig.add_hline(y=1.0, line_dash="dot",  line_color="gray", annotation_text="Avg severity = 1")
fig.update_layout(height=480)
fig.show()

## 4. Report Volume Over Time — Top 5 Topics (with PELT Changepoints)

In [6]:
top5_labels  = risks_sorted["label"].tolist()[:5]
top5_ids     = risks_sorted["Topic"].tolist()[:5]
label_to_id  = dict(zip(risks_sorted["label"], risks_sorted["Topic"]))
palette      = px.colors.qualitative.Plotly

fig = go.Figure()

for idx, (label, tid) in enumerate(zip(top5_labels, top5_ids)):
    td = trends[trends["topic_id"] == tid].sort_values("period")
    if td.empty:
        continue
    color = palette[idx % len(palette)]

    fig.add_trace(go.Scatter(
        x=td["period"], y=td["count"],
        mode="lines+markers",
        name=label,
        line=dict(color=color, width=2),
        marker=dict(size=5),
    ))

    cp_indices = changepoints.get(str(int(tid)), changepoints.get(int(tid), []))
    period_list = td["period"].tolist()
    for cp_i in cp_indices:
        if cp_i < len(period_list):
            fig.add_shape(
                type="line",
                x0=period_list[cp_i], x1=period_list[cp_i],
                y0=0, y1=1, yref="paper",
                line=dict(color=color, width=1.5, dash="dot"),
            )
            fig.add_annotation(
                x=period_list[cp_i], y=1.04, yref="paper",
                text="CP", font=dict(color=color, size=9),
                showarrow=False, xanchor="center",
            )

fig.update_layout(
    title="Report Volume Over Time — Top 5 Risk Topics (dotted lines = PELT changepoints)",
    xaxis_title="Period",
    yaxis_title="Report Count",
    height=470,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig.show()

## 5. Severity Trajectory — Top 5 Topics

In [7]:
fig = go.Figure()

for idx, (label, tid) in enumerate(zip(top5_labels, top5_ids)):
    td = trends[trends["topic_id"] == tid].sort_values("period")
    if td.empty or "avg_severity" not in td.columns:
        continue
    fig.add_trace(go.Scatter(
        x=td["period"], y=td["avg_severity"],
        mode="lines+markers",
        name=label,
        line=dict(color=palette[idx % len(palette)], width=2),
        marker=dict(size=5),
    ))

fig.add_hline(y=1.0, line_dash="dash", line_color="gray", annotation_text="Mean severity = 1")
fig.update_layout(
    title="Average Severity Trajectory — Top 5 Risk Topics",
    xaxis_title="Period",
    yaxis_title="Avg Severity (0 = none, 3 = fatal)",
    height=400,
    hovermode="x unified",
    yaxis=dict(range=[0, 3.2]),
)
fig.show()

## 6. Topic Deep-Dive: Single Topic Volume + Severity + Keywords

In [8]:
# Adjust this to any topic you want to inspect
selected_label = risks_sorted.iloc[0]["label"]
selected_tid   = risks_sorted.iloc[0]["Topic"]
row            = risks_sorted.iloc[0]
keywords       = _get_keywords(selected_tid)

print(f"Topic:        {selected_label}")
print(f"Risk Score:   {row['Risk_Score']:.3f}")
print(f"Growth Ratio: {row['Growth_Ratio']:.3f}")
print(f"Avg Severity: {row['Avg_Severity']:.3f}")
print(f"Reports:      {int(row['Count']):,}")
print(f"Changepoint:  {row['Recent_Changepoint']}")
print(f"Keywords:     {keywords[:12]}")

td = trends[trends["topic_id"] == selected_tid].sort_values("period")

# Volume bar chart
fig = go.Figure()
fig.add_trace(go.Bar(
    x=td["period"], y=td["count"],
    name="Report Count",
    marker_color="#3498db",
))
cp_indices  = changepoints.get(str(int(selected_tid)), changepoints.get(int(selected_tid), []))
period_list = td["period"].tolist()
for cp_i in cp_indices:
    if cp_i < len(period_list):
        fig.add_shape(
            type="line",
            x0=period_list[cp_i], x1=period_list[cp_i],
            y0=0, y1=1, yref="paper",
            line=dict(color="#e74c3c", width=2.5),
        )
        fig.add_annotation(
            x=period_list[cp_i], y=1.06, yref="paper",
            text="Changepoint",
            font=dict(color="#e74c3c", size=10),
            showarrow=False, xanchor="right",
        )
fig.update_layout(
    title=f"Report Volume — {selected_label}",
    xaxis_title="Period", yaxis_title="Count", height=340,
)
fig.show()

# Severity area chart
if "avg_severity" in td.columns:
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(
        x=td["period"], y=td["avg_severity"],
        mode="lines+markers",
        fill="tozeroy",
        fillcolor="rgba(231,76,60,0.15)",
        line=dict(color="#e74c3c", width=2),
        marker=dict(size=6),
        name="Avg Severity",
    ))
    fig2.add_hline(y=1.0, line_dash="dash", line_color="gray")
    fig2.update_layout(
        title=f"Severity Trend — {selected_label}",
        xaxis_title="Period", yaxis_title="Avg Severity",
        height=300,
        yaxis=dict(range=[0, 3.2]),
    )
    fig2.show()

Topic:        Tire Tires Gear Wheel
Risk Score:   4.788
Growth Ratio: 3.125
Avg Severity: 1.064
Reports:      78
Changepoint:  False
Keywords:     ['tire', 'tires', 'gear', 'wheel', 'assembly', 'maintenance', 'landing', 'main', 'landing gear', 'aircraft']


## 7. Full Risk Summary Table

In [9]:
display_df = risks_sorted[[
    "label", "Risk_Score", "Growth_Ratio", "Avg_Severity", "Count", "Recent_Changepoint"
]].copy()
display_df.columns = ["Topic", "Risk Score", "Growth Ratio", "Avg Severity", "Reports", "Recent Changepoint"]
for col in ["Risk Score", "Growth Ratio", "Avg Severity"]:
    display_df[col] = display_df[col].round(3)

print(display_df.set_index("Topic").to_string())

                                           Risk Score  Growth Ratio  Avg Severity  Reports  Recent Changepoint
Topic                                                                                                         
Tire Tires Gear Wheel                           4.788         3.125         1.064       78               False
Smoke Smell Odor Flight                         4.478         1.522         1.922     1760                True
Flaps Flap Edge Landing                         3.601         2.222         1.241      220               False
Electrical Alternator Power Failure             2.585         1.552         1.332      211               False
Gps Flight Jamming Reported                     2.201         1.946         0.262      862               False
Engine Fuel Flight Landing                      1.553         0.905         1.433     4702               False
Oxygen O2 Mask Crew Oxygen                      1.362         0.984         0.770      126               False
G

## 8. Severity Heatmap Over Time (all topics)

In [10]:
if "avg_severity" in trends.columns:
    pivot = (
        trends
        .merge(risks[["Topic", "label"]], left_on="topic_id", right_on="Topic", how="left")
        .pivot_table(index="label", columns="period", values="avg_severity", aggfunc="mean")
        .fillna(0)
    )
    # Keep top 20 topics by max severity for readability
    pivot = pivot.loc[pivot.max(axis=1).nlargest(20).index]

    fig = px.imshow(
        pivot.values,
        x=pivot.columns.tolist(),
        y=pivot.index.tolist(),
        color_continuous_scale="RdYlGn_r",
        zmin=0, zmax=2,
        title="Average Severity Heatmap — Top 20 Topics Over Time",
        labels={"x": "Period", "y": "Topic", "color": "Avg Severity"},
    )
    fig.update_layout(height=520, margin=dict(l=200))
    fig.show()

## 9. Changepoint Summary

In [11]:
cp_summary = []
for tid_str, cp_list in changepoints.items():
    try:
        tid     = int(tid_str)
        match   = risks[risks["Topic"] == tid]
        label   = match.iloc[0]["label"] if not match.empty else f"Topic {tid}"
        rs      = float(match.iloc[0]["Risk_Score"]) if not match.empty else 0.0
        cp_summary.append({"Topic": label, "Changepoints": len(cp_list), "Risk Score": rs})
    except Exception:
        pass

cp_df = pd.DataFrame(cp_summary).sort_values("Changepoints", ascending=False)

fig = px.bar(
    cp_df.head(20),
    x="Changepoints",
    y="Topic",
    orientation="h",
    color="Risk Score",
    color_continuous_scale="RdYlGn_r",
    title="Number of PELT Changepoints per Topic (top 20)",
    labels={"Changepoints": "Changepoint Count", "Topic": ""},
)
fig.update_yaxes(autorange="reversed")
fig.update_layout(height=max(350, min(20, len(cp_df)) * 28), margin=dict(l=200))
fig.show()